# 02 ~ Experimentos Comparativos (R0, R1, R2, C0 y Modelo Alterno)

Esta libreta implementa las variantes experimentales exigidas por el enunciado usando el módulo compartido `rag_core.py`, manteniendo fijos el corpus, las preguntas, la temperatura (`0.0`), el límite de salida (`500` tokens) y el límite máximo de contexto (`6000` caracteres).

## Hipótesis Previas (antes de ejecutar las pruebas finales)
1. **R1 vs R0 (Fragmentación por sección/oración vs tamaño fijo):** Al evitar cortes arbitrarios a mitad de oración (máx. 500 chars por unidad/oración), se espera que **R1** conserve mejor las reglas completas (ej. umbrales o pasos de un playbook) sin arrastrar fragmentos partidos.
2. **R2 vs R0 (Recuperación híbrida Densa + BM25 con mismo `k=4`):** Al combinar similitud semántica con coincidencia léxica BM25, se espera que **R2** recupere con mayor precisión pasajes que contienen identificadores exactos (`PB-01`, `INC-2026-019`, puertos `2222`/`5432`, `WORM`, `DPO`).
3. **C0 vs R0 (Control sin RAG):** Sin acceso al corpus de NovaCloud Corp, se espera que **C0** falle o responda de forma genérica ante preguntas de políticas/SLAs internos, aunque pueda acertar conceptos generales de ciberseguridad si no se abstiene.
4. **R0_ALT (Embeddings alterno + LLM distinto):** Usando `all-MiniLM-L6-v2` (entrenado principalmente en inglés) y otro LLM (`qwen/qwen-2.5-72b-instruct`), se espera observar si la calidad de recuperación en español disminuye frente a `paraphrase-multilingual-MiniLM-L12-v2`.

In [1]:
import os
import numpy as np
from openai import OpenAI
from sentence_transformers import SentenceTransformer

from rag_core import (
    cargar_env_local,
    resolver_ruta,
    cargar_corpus,
    fragmentar_tamano_fijo,
    fragmentar_por_seccion,
    construir_indice_bm25,
    recuperar_denso,
    recuperar_hibrido,
    consultar_rag_con_recuperados,
    consultar_control_c0,
)

# ----------------------------------------------------------------------
# PARÁMETROS FIJOS Y EXPERIMENTALES
# ----------------------------------------------------------------------
CARPETA_CORPUS = "corpus"
TOP_K = 4
TEMPERATURA = 0.0
MAX_TOKENS_SALIDA = 500
MAX_CONTEXTO_CHARS = 6000

# R0 (Base)
CHUNK_SIZE_R0 = 400
OVERLAP_R0 = 60
MODELO_EMB_R0 = "paraphrase-multilingual-MiniLM-L12-v2"
MODELO_LLM_BASE = "deepseek/deepseek-chat"

# R1 (Fragmentación por unidad/sección con límite similar)
MAX_CHARS_R1 = 500

# R2 (Híbrida: 60% Densa + 40% BM25, conservando k=4)
PESO_DENSO_R2 = 0.6

# Experimento alterno sobre R0
MODELO_EMB_ALT = "all-MiniLM-L6-v2"
MODELO_LLM_ALT = "qwen/qwen-2.5-72b-instruct"

# Cargar .env si existe y configurar cliente OpenRouter
cargar_env_local()
api_key_openrouter = os.getenv("API_KEY") or os.getenv("OPENROUTER_API_KEY")
client = OpenAI(
    api_key=api_key_openrouter,
    base_url="https://openrouter.ai/api/v1",
    default_headers={"HTTP-Referer": "http://localhost", "X-Title": "SOC RAG Experimentos"},
) if api_key_openrouter else None

if client is None:
    print("AVISO: Define API_KEY o OPENROUTER_API_KEY antes de ejecutar consultas al LLM.")

c:\Users\sebas\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Construcción de Índices (R0, R1, R2 y R0_ALT)

In [2]:
unidades = cargar_corpus(CARPETA_CORPUS)

# R0 y R2 comparten fragmentos base (400 chars, 60 overlap) y embeddings R0
embedder_r0 = SentenceTransformer(MODELO_EMB_R0)
chunks_r0 = fragmentar_tamano_fijo(unidades, CHUNK_SIZE_R0, OVERLAP_R0)
emb_r0 = embedder_r0.encode([c["texto"] for c in chunks_r0], normalize_embeddings=True)

# R1: Fragmentación por sección/oración
chunks_r1 = fragmentar_por_seccion(unidades, max_chars=MAX_CHARS_R1)
emb_r1 = embedder_r0.encode([c["texto"] for c in chunks_r1], normalize_embeddings=True)

# R2: Índice BM25 sobre los mismos chunks de R0
bm25_r2 = construir_indice_bm25(chunks_r0)

# R0_ALT: Mismos chunks de R0, reindexados con otro modelo de embeddings
embedder_alt = SentenceTransformer(MODELO_EMB_ALT)
emb_alt = embedder_alt.encode([c["texto"] for c in chunks_r0], normalize_embeddings=True)

print(f"R0/R2 chunks: {len(chunks_r0)} | R1 chunks: {len(chunks_r1)}")
print(f"Embeddings R0: {emb_r0.shape} ({MODELO_EMB_R0})")
print(f"Embeddings ALT: {emb_alt.shape} ({MODELO_EMB_ALT})")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14364.53it/s]


R0/R2 chunks: 56 | R1 chunks: 51
Embeddings R0: (56, 384) (paraphrase-multilingual-MiniLM-L12-v2)
Embeddings ALT: (56, 384) (all-MiniLM-L6-v2)


## 2. Ejecutor Unificado de Configuraciones (R0, R1, R2, C0, R0_ALT)

In [3]:
def ejecutar_configuracion(config, pregunta):
    """Ejecuta una pregunta bajo la configuración indicada: R0, R1, R2, C0 o R0_ALT."""
    if config == "R0":
        rec = recuperar_denso(pregunta, chunks_r0, emb_r0, embedder_r0, top_k=TOP_K)
        res = consultar_rag_con_recuperados(client, pregunta, rec, MODELO_LLM_BASE, TEMPERATURA, MAX_TOKENS_SALIDA, MAX_CONTEXTO_CHARS)
    elif config == "R1":
        rec = recuperar_denso(pregunta, chunks_r1, emb_r1, embedder_r0, top_k=TOP_K)
        res = consultar_rag_con_recuperados(client, pregunta, rec, MODELO_LLM_BASE, TEMPERATURA, MAX_TOKENS_SALIDA, MAX_CONTEXTO_CHARS)
    elif config == "R2":
        rec = recuperar_hibrido(pregunta, chunks_r0, emb_r0, embedder_r0, bm25_r2, top_k=TOP_K, peso_denso=PESO_DENSO_R2)
        res = consultar_rag_con_recuperados(client, pregunta, rec, MODELO_LLM_BASE, TEMPERATURA, MAX_TOKENS_SALIDA, MAX_CONTEXTO_CHARS)
    elif config == "C0":
        res = consultar_control_c0(client, pregunta, MODELO_LLM_BASE, TEMPERATURA, MAX_TOKENS_SALIDA)
    elif config == "R0_ALT":
        rec = recuperar_denso(pregunta, chunks_r0, emb_alt, embedder_alt, top_k=TOP_K)
        res = consultar_rag_con_recuperados(client, pregunta, rec, MODELO_LLM_ALT, TEMPERATURA, MAX_TOKENS_SALIDA, MAX_CONTEXTO_CHARS)
    else:
        raise ValueError(f"Configuración desconocida: {config}")

    res["config"] = config
    return res


# Prueba rápida local de recuperación (sin gastar tokens de LLM) para verificar R0 vs R1 vs R2 vs R0_ALT
pregunta_demo = "¿Qué se hizo en el incidente INC-2026-019 contra el puerto 2222?"
print(f"Prueba de recuperación para: {pregunta_demo}\n")
for nombre_cfg, rec in [
    ("R0 (Denso)", recuperar_denso(pregunta_demo, chunks_r0, emb_r0, embedder_r0, TOP_K)),
    ("R1 (Sección)", recuperar_denso(pregunta_demo, chunks_r1, emb_r1, embedder_r0, TOP_K)),
    ("R2 (Híbrido)", recuperar_hibrido(pregunta_demo, chunks_r0, emb_r0, embedder_r0, bm25_r2, TOP_K, PESO_DENSO_R2)),
    ("R0_ALT (MiniLM-EN)", recuperar_denso(pregunta_demo, chunks_r0, emb_alt, embedder_alt, TOP_K)),
]:
    top1 = rec[0]
    print(f"  {nombre_cfg:18s} -> Top 1: [{top1['archivo']} | {top1['localizador']}] (score={top1['score']:.3f})")

Prueba de recuperación para: ¿Qué se hizo en el incidente INC-2026-019 contra el puerto 2222?

  R0 (Denso)         -> Top 1: [Registro_Incidentes_Resueltos.md | Sección 'INC-2026-027 — Exposición temporal de puerto PostgreSQL en ambiente staging' (Líneas 37-42) [Parte 1]] (score=0.501)
  R1 (Sección)       -> Top 1: [Registro_Incidentes_Resueltos.md | Sección 'INC-2026-019 — Ataque de fuerza bruta contra bastión SSH' (Líneas 17-26) [Bloque 1]] (score=0.494)
  R2 (Híbrido)       -> Top 1: [Registro_Incidentes_Resueltos.md | Sección 'INC-2026-019 — Ataque de fuerza bruta contra bastión SSH' (Líneas 17-26) [Parte 1]] (score=0.990)
  R0_ALT (MiniLM-EN) -> Top 1: [Registro_Incidentes_Resueltos.md | Sección 'INC-2026-027 — Exposición temporal de puerto PostgreSQL en ambiente staging' (Líneas 37-42) [Parte 1]] (score=0.564)


## 3. Protocolo de Evaluación ~ Banco Oficial de 9 Preguntas
Se fijan 9 preguntas antes de las corridas finales:
* **3 directas** (1 documento): `Q1`, `Q2`, `Q3`.
* **3 combinadas** ($\ge 2$ documentos): `Q4`, `Q5`, `Q6`.
* **3 sin respuesta en el corpus** (abstención): `Q7`, `Q8`, `Q9`.

In [4]:
BANCO_PREGUNTAS = [
    {
        "id": "Q1",
        "tipo": "directa",
        "pregunta": "¿Cuál es la longitud mínima de contraseña para cuentas administrativas, qué métodos MFA están permitidos y por cuánto tiempo se bloquea la cuenta tras intentos fallidos?",
        "respuesta_esperada": "Longitud mínima de 16 caracteres para cuentas administrativas; MFA permitido únicamente con llaves físicas FIDO2/WebAuthn o apps TOTP (prohibido SMS/llamadas); bloqueo automático de 15 minutos tras 5 intentos fallidos.",
        "evidencia_requerida": [
            {"archivo": "Politica_Seguridad_y_Accesos.pdf", "localizador": "Página 2", "claves": ["16", "FIDO2", "TOTP", "15 minutos", "5 intentos"]},
        ],
    },
    {
        "id": "Q2",
        "tipo": "directa",
        "pregunta": "¿En qué puerto debe escuchar el servicio SSH en los servidores Linux, qué pasa con el usuario root y qué algoritmos de llave pública están permitidos?",
        "respuesta_esperada": "El servicio SSH debe escuchar en el puerto 2222 (prohibido el 22); se deshabilita el acceso directo de root (PermitRootLogin no) y contraseñas; solo se permiten llaves Ed25519 o RSA de 4096 bits.",
        "evidencia_requerida": [
            {"archivo": "Guia_Hardening_Red_y_Servidores.pdf", "localizador": "Página 1", "claves": ["2222", "PermitRootLogin no", "Ed25519", "4096"]},
        ],
    },
    {
        "id": "Q3",
        "tipo": "directa",
        "pregunta": "¿Qué pasos obligatorios indica el playbook PB-03 para un ataque de fuerza bruta SSH o VPN, incluyendo el umbral de intentos y la duración del bloqueo?",
        "respuesta_esperada": "Verificar si una misma IP supera 20 intentos fallidos en 5 minutos; comprobar si hubo inicio de sesión exitoso posterior; si no hubo éxito, bloquear la IP en firewall perimetral por 24 horas (P3); si hubo éxito, escalar a P2/P1 y revocar llaves o certificados.",
        "evidencia_requerida": [
            {"archivo": "Playbook_Respuesta_Incidentes.md", "localizador": "PB-03", "claves": ["20", "5 minutos", "24 horas"]},
        ],
    },
    {
        "id": "Q4",
        "tipo": "combinada",
        "pregunta": "Si un analista recibe una alerta de sospecha de ransomware, ¿por qué no debe apagar el equipo según el playbook y en qué tiempos máximos de primera respuesta, contención y notificación al CISO debe atenderse el incidente?",
        "respuesta_esperada": "Según PB-02 no se debe apagar para preservar la memoria volátil (RAM) y posibles llaves de cifrado; todo ransomware activo es P1 (Crítico), por lo que según la matriz SLA exige primera respuesta en 15 minutos, contención en 1 hora y notificación inmediata al CISO.",
        "evidencia_requerida": [
            {"archivo": "Playbook_Respuesta_Incidentes.md", "localizador": "PB-02", "claves": ["RAM", "apagar"]},
            {"archivo": "Matriz_SLA_y_Escalamiento.docx", "localizador": "Tabla 1 / Sección 2", "claves": ["15 minutos", "1 hora", "CISO"]},
        ],
    },
    {
        "id": "Q5",
        "tipo": "combinada",
        "pregunta": "En el incidente INC-2026-019 contra el bastión SSH, ¿qué configuración de hardening impidió la intrusión y a qué origen de red restringe la guía técnica el acceso al puerto 2222?",
        "respuesta_esperada": "En INC-2026-019 se confirmó PasswordAuthentication no, PermitRootLogin no y llaves Ed25519; según la guía de hardening, el puerto 2222 está restringido únicamente al origen VPN Corporativa.",
        "evidencia_requerida": [
            {"archivo": "Registro_Incidentes_Resueltos.md", "localizador": "INC-2026-019", "claves": ["PasswordAuthentication no", "Ed25519"]},
            {"archivo": "Guia_Hardening_Red_y_Servidores.pdf", "localizador": "Página 1", "claves": ["2222", "VPN Corporativa"]},
        ],
    },
    {
        "id": "Q6",
        "tipo": "combinada",
        "pregunta": "Si un ataque de phishing compromete una cuenta con acceso a datos clasificados como Restringido, ¿a qué severidad se reclasifica según el playbook y en qué plazo máximo debe notificarse al DPO?",
        "respuesta_esperada": "Según PB-01 paso 5, se reclasifica automáticamente a severidad P1; según la Matriz de SLA, si hay posible exposición de datos Restringido debe notificarse al DPO en un plazo máximo de 2 horas desde la confirmación.",
        "evidencia_requerida": [
            {"archivo": "Playbook_Respuesta_Incidentes.md", "localizador": "PB-01", "claves": ["P1", "Restringido"]},
            {"archivo": "Matriz_SLA_y_Escalamiento.docx", "localizador": "Sección 2", "claves": ["DPO", "2 horas"]},
        ],
    },
    {
        "id": "Q7",
        "tipo": "sin_respuesta",
        "pregunta": "¿Cuál es el proveedor específico de EDR y la marca del firewall perimetral contratados por NovaCloud Corp?",
        "respuesta_esperada": "Abstención: el corpus menciona genéricamente 'el EDR' y 'firewall perimetral', pero nunca especifica proveedores ni marcas comerciales.",
        "evidencia_requerida": [],
    },
    {
        "id": "Q8",
        "tipo": "sin_respuesta",
        "pregunta": "¿Cuál es el presupuesto anual asignado al SOC de NovaCloud Corp y cuántas personas trabajan en el turno nocturno?",
        "respuesta_esperada": "Abstención: los documentos no contienen información financiera, presupuestal ni de dotación de turnos del personal.",
        "evidencia_requerida": [],
    },
    {
        "id": "Q9",
        "tipo": "sin_respuesta",
        "pregunta": "¿Qué procedimiento debe seguir un empleado si pierde su teléfono móvil corporativo durante un viaje internacional?",
        "respuesta_esperada": "Abstención: el corpus no incluye protocolos de pérdida o robo de teléfonos móviles ni políticas para viajes internacionales.",
        "evidencia_requerida": [],
    },
]
print(f"Banco oficial cargado: {len(BANCO_PREGUNTAS)} preguntas.")

Banco oficial cargado: 9 preguntas.


## 4. Ejecución de las 45 Corridas (9 preguntas × 5 configuraciones) y Exportación
Ejecuta todas las configuraciones (`R0`, `R1`, `R2`, `C0`, `R0_ALT`), reintenta si ocurre un fallo transitorio de red, calcula automáticamente la **cobertura de evidencia** (recuperada y en contexto) y guarda los artefactos en `../resultados/`.

In [5]:
import csv
import json
import time

CONFIGURACIONES = ["R0", "R1", "R2", "C0", "R0_ALT"]


def verificar_cobertura(evidencia_req, recuperados, contexto_enviado):
    if not evidencia_req:
        return None, None
    texto_rec = " ".join(f"{r['archivo']} {r['texto']}" for r in recuperados).lower()
    ctx_lower = contexto_enviado.lower()

    def _cumple(texto_objetivo):
        for ev in evidencia_req:
            if ev["archivo"].lower() not in texto_objetivo:
                return False
            for clave in ev["claves"]:
                if clave.lower() not in texto_objetivo:
                    return False
        return True

    return _cumple(texto_rec), _cumple(ctx_lower)


def detectar_abstencion_preliminar(respuesta):
    marcadores = [
        "no encuentro evidencia suficiente",
        "no cuento con información",
        "no tengo acceso",
        "no figura en",
        "no se menciona",
        "no puedo responder",
        "carezco de información",
    ]
    r_low = respuesta.lower()
    return any(m in r_low for m in marcadores)


resultados_completos = []
fallos_registrados = []

for cfg in CONFIGURACIONES:
    print(f"\n>>> Ejecutando configuración: {cfg}")
    for item in BANCO_PREGUNTAS:
        exito = False
        intentos = 0
        while not exito and intentos < 3:
            intentos += 1
            try:
                res = ejecutar_configuracion(cfg, item["pregunta"])
                exito = True
            except Exception as exc:
                fallos_registrados.append({
                    "config": cfg, "id_pregunta": item["id"], "intento": intentos, "error": str(exc)
                })
                print(f"  [!] Fallo en {cfg} {item['id']} (intento {intentos}): {exc}")
                time.sleep(2)

        if not exito:
            continue

        cob_rec, cob_ctx = verificar_cobertura(
            item["evidencia_requerida"], res["recuperados"], res["contexto_enviado"]
        )
        abst_auto = detectar_abstencion_preliminar(res["respuesta"]) if item["tipo"] == "sin_respuesta" else None

        registro = {
            "config": cfg,
            "id_pregunta": item["id"],
            "tipo": item["tipo"],
            "pregunta": item["pregunta"],
            "respuesta_esperada": item["respuesta_esperada"],
            "respuesta_modelo": res["respuesta"],
            "tiempo_seg": res["tiempo_seg"],
            "contexto_recortado": res["contexto_recortado"],
            "cobertura_recuperada": cob_rec,
            "cobertura_en_contexto": cob_ctx,
            "abstencion_preliminar": abst_auto,
            "fragmentos_recuperados": [
                {"rank": r["rank"], "score": round(r["score"], 4), "archivo": r["archivo"],
                 "localizador": r["localizador"], "texto": r["texto"]}
                for r in res["recuperados"]
            ],
            "contexto_enviado": res["contexto_enviado"],
        }
        resultados_completos.append(registro)
        print(f"  {item['id']} ({item['tipo']:13s}) | {res['tiempo_seg']:4.2f}s | cob_ctx={cob_ctx} | abst={abst_auto}")

# Guardar paquete de evaluación en resultados/
ruta_resultados = resolver_ruta("resultados")
os.makedirs(ruta_resultados, exist_ok=True)

ruta_json = os.path.join(ruta_resultados, "resultados_corridas.json")
with open(ruta_json, "w", encoding="utf-8") as f:
    json.dump({"corridas": resultados_completos, "fallos": fallos_registrados}, f, ensure_ascii=False, indent=2)

ruta_csv = os.path.join(ruta_resultados, "resultados_evaluacion.csv")
campos = [
    "config", "id_pregunta", "tipo", "tiempo_seg", "contexto_recortado",
    "cobertura_recuperada", "cobertura_en_contexto", "abstencion_preliminar",
    "respuesta_modelo", "respuesta_esperada"
]
with open(ruta_csv, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=campos)
    writer.writeheader()
    for r in resultados_completos:
        writer.writerow({k: r[k] for k in campos})

print(f"\nTotal corridas guardadas: {len(resultados_completos)} (fallos recuperados: {len(fallos_registrados)})")
print(f"JSON: {ruta_json}")
print(f"CSV : {ruta_csv}")


>>> Ejecutando configuración: R0
  Q1 (directa      ) | 9.10s | cob_ctx=False | abst=None
  Q2 (directa      ) | 2.64s | cob_ctx=True | abst=None
  Q3 (directa      ) | 5.66s | cob_ctx=True | abst=None
  Q4 (combinada    ) | 3.36s | cob_ctx=True | abst=None
  Q5 (combinada    ) | 4.86s | cob_ctx=False | abst=None
  Q6 (combinada    ) | 3.50s | cob_ctx=True | abst=None
  Q7 (sin_respuesta) | 1.15s | cob_ctx=None | abst=True
  Q8 (sin_respuesta) | 1.90s | cob_ctx=None | abst=True
  Q9 (sin_respuesta) | 4.95s | cob_ctx=None | abst=True

>>> Ejecutando configuración: R1
  Q1 (directa      ) | 2.32s | cob_ctx=True | abst=None
  Q2 (directa      ) | 1.61s | cob_ctx=False | abst=None
  Q3 (directa      ) | 5.67s | cob_ctx=True | abst=None
  Q4 (combinada    ) | 5.66s | cob_ctx=False | abst=None
  Q5 (combinada    ) | 5.72s | cob_ctx=False | abst=None
  Q6 (combinada    ) | 5.21s | cob_ctx=False | abst=None
  Q7 (sin_respuesta) | 1.08s | cob_ctx=None | abst=True
  Q8 (sin_respuesta) | 2.11s |